In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots
from statsmodels.api import OLS
import sklearn.model_selection as skm
import sklearn.linear_model as skl
from sklearn.preprocessing import StandardScaler
from ISLP import load_data
from ISLP.models import ModelSpec as MS
from functools import partial

from sklearn.pipeline import Pipeline 
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from functools import partial 


from ISLP.models import (Stepwise, 
                        sklearn_selected, 
                        sklearn_selection_path) 

from l0bnb import fit_path
from utils import nCp


In [ ]:
Hitters = load_data("Hitters") 

print(np.isnan(Hitters["Salary"]).sum()) 

Hitters = Hitters.dropna()
design = MS(Hitters.columns.drop("Salary")).fit(Hitters)

Y = Hitters["Salary"].values
X = design.transform(Hitters)
sigma2 = OLS(Y, X).fit().scale

In [ ]:
negCp = partial(nCp, sigma2)

strategy = Stepwise.first_peak(design, 
                               direction='forward', 
                               max_terms=len(design.terms))

hitters_MSE = sklearn_selected (OLS, strategy) 

hitters_MSE.fit(Hitters , Y)
# no scoring argument and all covariates are selected) 
hitters_MSE.selected_state_

In [ ]:
# add a scoring rule
hitters_MSE = sklearn_selected (OLS, strategy, scoring=negCp) 

hitters_MSE.fit(Hitters , Y)
# no scoring argument and all covariates are selected) 
hitters_MSE.selected_state_

In [ ]:
strategy = Stepwise.fixed_steps(design, len(design.terms),direction='forward')
full_path = sklearn_selection_path(OLS , strategy)

full_path.fit(Hitters, Y)
Yhat_in = full_path.predict(Hitters)

In [ ]:
 mse_fig , ax = subplots(figsize =(8 ,8))
insample_mse = (( Yhat_in - Y[:,None ]) **2).mean(0)
n_steps = insample_mse.shape [0]
ax.plot(np.arange(n_steps), insample_mse, 'k', # color black
        label='In -sample ')
ax.set_ylabel('MSE', fontsize =20)
ax.set_xlabel('# steps of forward stepwise ', fontsize =20)
ax.set_xticks(np.arange(n_steps)[::2])
ax.legend ()
ax.set_ylim([50000, 250000]) ; 

In [ ]:
K = 5 
kfold = skm.KFold(K, random_state=0, shuffle=True) 

Yhat_cv = skm.cross_val_predict(full_path, 
                                Hitters, 
                                Y, 
                                cv=kfold)

In [ ]:
Yhat_cv.shape, Hitters.shape, Y.shape

In [ ]:
cv_mse = []
for train_idx , test_idx in kfold.split(Y):
    errors = (Yhat_cv[test_idx] - Y[test_idx, None ])**2
    
    cv_mse.append(errors.mean (0)) # column means
    
cv_mse = np.array(cv_mse).T
cv_mse.shape

In [ ]:
cv_mse.mean(1).shape

In [ ]:
ax.errorbar(np.arange(n_steps),cv_mse.mean(1),
cv_mse.std(1) / np.sqrt(K),
label='Cross -validated', c='r') # color red
ax.set_ylim([50000 ,250000])
ax.legend()
mse_fig

In [ ]:
# Example on the broadcasting trick being used
yy = np.arange(1,11) 
some_idx = [1, 3, 5]  
yyy = yy[some_idx, None]

z = np.zeros((7, 4))
print(yy[some_idx])
z[some_idx]  - yyy

In [ ]:
validation = skm.ShuffleSplit(n_splits=1, test_size =0.2, random_state= 0) 

for train_idx , test_idx in validation.split(Y):
    full_path.fit(Hitters.iloc[train_idx], Y[train_idx ])
    Yhat_val = full_path.predict(Hitters.iloc[test_idx ])
    errors = (Yhat_val - Y[test_idx ,None ])**2
    validation_mse = errors.mean (0) 


In [ ]:
ax.plot(np.arange(n_steps), validation_mse , 'b--', # color blue , broken line
        label='Validation') 

ax.set_xticks(np.arange(n_steps)[::2])
ax.set_ylim ([50000 ,250000])
ax.legend()
mse_fig

### Best subset selection 

In [ ]:
# not working 
# D = design.fit_transform(Hitters).drop('intercept', axis=1) 
# X = np.asarray(D)
# path = fit_path(X,Y, max_nonzeros=X.shape [1]) ;

#### Ridge and Lasso 

In [ ]:
Xs = X - X.mean(0)[None, :]
X_scale = X.std(0)
Xs = Xs / X_scale[None, :]
lambdas = 10** np.linspace (8, -2, 100) / Y.std()
soln_array = skl.ElasticNet.path(Xs, Y, l1_ratio =0., alphas=lambdas)[1]


In [ ]:
soln_path = pd.DataFrame(soln_array.T,
                        columns=D.columns, 
                         index=-np.log(lambdas))

soln_path.index.name = 'negative log(lambda)'
print(Y.std())
soln_path

In [ ]:
path_fig , ax = subplots(figsize =(8 ,8))
soln_path.plot(ax=ax , legend=False)
ax.set_xlabel(r'$- \log(\lambda)$', fontsize =20) 
ax.set_ylabel('Standardized coefficients ', fontsize =20)
ax.legend(loc='upper left');

In [ ]:
beta_hat = soln_path.loc[soln_path.index [39]]
lambdas [39], beta_hat

In [ ]:
# the nice and modern way to scale and fit 
ridge = skl.ElasticNet(alpha=lambdas [59], l1_ratio =0, max_iter=int(1e4))
scaler = StandardScaler(with_mean=True , with_std=True)
pipe = Pipeline(steps =[('scaler', scaler), ('ridge', ridge)])
pipe.fit(X, Y);

In [ ]:
# here is an example of grid serach with cross validation 
param_grid = {'ridge__alpha': lambdas}
grid = skm.GridSearchCV(pipe, param_grid, cv=validation, scoring='neg_mean_squared_error')

grid.fit(X, Y)
grid.best_params_['ridge__alpha']
grid.best_estimator_

In [ ]:
lassoCV = skl.ElasticNetCV(l1_ratio =1, cv=kfold)
pipeCV = Pipeline(steps =[('scaler ', scaler), ('lasso ', lassoCV)])
pipeCV.fit(X, Y)
tuned_lasso = pipeCV.named_steps['lasso ']
tuned_lasso.alpha_

In [ ]:
lambdas , soln_array = skl.Lasso.path(Xs, Y, 
                                      l1_ratio =1, n_alphas =100) [:2]
soln_path = pd.DataFrame(soln_array.T, columns=D.columns, index=-np.log(lambdas))

path_fig , ax = subplots(figsize =(8 ,8))
soln_path.plot(ax=ax , legend=False)
ax.legend(loc='upper left')
ax.set_xlabel(r'$-\log(\ lambda)$', fontsize =20)
ax.set_ylabel(r'Standardized coefficiients ', fontsize =20);

In [ ]:
np.min(tuned_lasso.mse_path_.mean (1))

In [ ]:
lassoCV_fig , ax = subplots(figsize =(8 ,8))
ax.errorbar(-np.log(tuned_lasso.alphas_), tuned_lasso.mse_path_.mean (1),
            yerr=tuned_lasso.mse_path_.std (1) / np.sqrt(K))

ax.axvline(-np.log(tuned_lasso.alpha_), c='k', ls='--')
ax.set_ylim ([50000 ,250000])
ax.set_xlabel('$-\log(\ lambda)$', fontsize =20)
ax.set_ylabel('Cross -validated MSE', fontsize =20);

In [ ]:
pca = PCA(n_components =2)
linreg = skl.LinearRegression ()
pipe = Pipeline ([('pca', pca),
('linreg ', linreg)])
pipe.fit(X, Y)
pipe.named_steps['linreg ']. coef_

In [ ]:
pipe = Pipeline ([('scaler ', scaler),
('pca', pca),
('linreg ', linreg)])
pipe.fit(X, Y)
pipe.named_steps['linreg ']. coef_

In [ ]:
param_grid = {'pca__n_components': range (1, 20)}
grid = skm.GridSearchCV(pipe, param_grid, cv=kfold, scoring='neg_mean_squared_error')
grid.fit(X, Y)

#### Q8

In [ ]:
# Ex 8

rng = np.random.RandomState(123) 
n = 100 

X = rng.randn(n) 
epsilon = rng.randn(n) 

bettas = np.array([50, 44., 97, 5.])  

Xs = np.column_stack([X, X**2, X**3])

Y = np.dot(Xs, bettas[1:])  + bettas[0] + epsilon

plt.scatter(X, Y)


In [ ]:
X2 = np.column_stack([X**i for i in range(1, 11)])

X2 = pd.DataFrame(data=X2, columns=[f"X{i}" for i in range(1,11)])

X2['Y'] = Y
X2.head()

In [ ]:
design = MS(X2.columns.drop('Y')).fit(X2)

X3 = design.transform(X2).drop('intercept', axis=1) 
sigma2 = OLS(Y, X3).fit().scale
sigma2

neg_Cp = partial(nCp,sigma2) 

strategy = Stepwise.first_peak(design,
                               direction='forward',
                               max_terms=len(design.terms))

model_forward = sklearn_selected(OLS,
                              strategy,
                              scoring=neg_Cp)

model_forward.fit(X3, Y)
model_forward.selected_state_


In [ ]:
print(f"Actual bettas are: {bettas}")

In [ ]:
model_forward.results_.summary()

In [ ]:
# Backward selection 
strategy = Stepwise.first_peak(design,
                               direction='backwards',
                               max_terms=len(design.terms))

model_backwards = sklearn_selected(OLS,
                              strategy,
                              scoring=neg_Cp)

model_backwards.fit(X3, Y)
model_backwards.selected_state_


In [ ]:
model_backwards.results_.summary()

#### Q9

In [ ]:
college = load_data("College") 

college.head()

In [ ]:
college.info()

In [ ]:
college.describe(include="all")

In [ ]:
design = MS(college.columns.drop('Apps'))

X = design.fit_transform(college)
y = college['Apps']

X_train, X_test, y_train, y_test = skm.train_test_split(X, y, test_size=0.3, random_state=1)

In [ ]:
results = OLS(y_train, X_train).fit()
pred = results.predict(X_test)
ols_MSE = np.mean((pred - y_test)**2)
ols_MSE

In [ ]:
# We have to drop the intercept when fitting RidgeCV and LassoCV since they add an intercept by default.
X_train = X_train.drop('intercept', axis=1)
X_test = X_test.drop('intercept', axis=1)

K = 5 
lambdas = np.linspace(0, 50, 100) 
inner_kfold = skm.KFold(n_splits=K,
               shuffle=True,
               random_state=2)
scaler = StandardScaler()

ridgeCV = skl.RidgeCV(alphas=lambdas,
                      cv=inner_kfold)
pipeCV = Pipeline([('scaler', scaler), ('ridge', ridgeCV)])

pipeCV.fit(X_train, y_train)

In [ ]:
pipeCV.named_steps["ridge"].alpha_

In [ ]:
pipeCV.named_steps['ridge'].intercept_

In [ ]:
pipeCV.named_steps['ridge'].coef_

In [ ]:
K = 5 

inner_kfold = skm.KFold(n_splits=K,
               shuffle=True,
               random_state=2)

scaler = StandardScaler()

lassoCV = skl.ElasticNetCV(l1_ratio=1,
                           cv=inner_kfold)

pipeCV = Pipeline([('scaler', scaler),
                ('lasso', lassoCV)])

pipeCV.fit(X_train, y_train)
# ....and continue as above 

In [ ]:
K = 5 

inner_kfold = skm.KFold(n_splits=K,
               shuffle=True,
               random_state=2)

scaler = StandardScaler()

lassoCV = skl.ElasticNetCV(l1_ratio=1,
                           cv=inner_kfold)

pipeCV = Pipeline([('scaler', scaler),
                ('lasso', lassoCV)])

pipeCV.fit(X_train, y_train)
# ....and continue as above 

In [ ]:
tuned_lasso = pipeCV.named_steps['lasso']
print(tuned_lasso.alpha_)

pred = pipeCV.predict(X_test)
lasso_MSE = np.mean((pred - y_test)**2)
lasso_MSE

In [ ]:
print(f"The number of non zero coefs out of lasso is: {sum(tuned_lasso.coef_ != 0)}")

In [ ]:
# alternatively 
K = 5 
scaler = StandardScaler() 
pipe = Pipeline([("scaler", scaler), 
                 ("lasso", skl.ElasticNet(l1_ratio=1.))
                ]) 
lambdas = np.linspace(0.1, 50, 100) 

kfold = skm.KFold(n_splits=K, 
                  shuffle=True, 
                  random_state=2)

param_grid = {"lasso__alpha": lambdas} 

grid = skm.GridSearchCV(pipe, 
                        param_grid, 
                        cv=kfold, 
                        scoring='neg_mean_squared_error')

grid.fit(X_train, y_train)

In [ ]:
grid.best_params_